In [1]:
!pip install -q langchain-openai langchain playwright beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.0 MB/s eta 0:00:00


In [2]:
!pip install playwright


In [3]:
!playwright install


167.7 MiB [] 0% 127.6s167.7 MiB [] 0% 33.3s167.7 MiB [] 0% 14.1s167.7 MiB [] 0% 7.4s167.7 MiB [] 1% 5.5s167.7 MiB [] 1% 4.0s167.7 MiB [] 2% 3.4s167.7 MiB [] 3% 3.2s167.7 MiB [] 3% 3.3s167.7 MiB [] 4% 3.3s167.7 MiB [] 5% 3.4s167.7 MiB [] 5% 3.7s167.7 MiB [] 5% 3.9s167.7 MiB [] 5% 4.0s167.7 MiB [] 6% 4.2s167.7 MiB [] 6% 4.4s167.7 MiB [] 6% 4.5s167.7 MiB [] 6% 4.7s167.7 MiB [] 7% 4.6s167.7 MiB [] 8% 4.4s167.7 MiB [] 8% 4.3s167.7 MiB [] 9% 4.2s167.7 MiB [] 9% 4.3s167.7 MiB [] 9% 4.2s167.7 MiB [] 9% 4.3s167.7 MiB [] 10% 4.4s167.7 MiB [] 10% 4.5s167.7 MiB [] 11% 4.4s167.7 MiB [] 11% 4.3s167.7 MiB [] 11% 4.4s167.7 MiB [] 12% 4.4s167.7 MiB [] 12% 4.5s167.7 MiB [] 12% 4.4s167.7 MiB [] 13% 4.3s167.7 MiB [] 13% 4.5s167.7 MiB [] 13% 4.4s167.7 MiB [] 14% 4.3s167.7 MiB [] 14% 4.5s167.7 MiB [] 14% 4.6s167.7 MiB [] 15% 4.6s167.7 MiB [] 15% 4.5s167.7 MiB [] 16% 4.5s167.7 MiB [] 17% 4.5s167.7 MiB [] 18% 4.4s167.7 MiB [] 18% 4.3s167.7 MiB [] 19% 4.3s167.7 MiB [] 20% 4.2s167.7 MiB [] 21% 4.1s167.7 MiB [] 

In [4]:
!pip install aiohttp

In [9]:
import nest_asyncio
import os
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

nest_asyncio.apply()
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 …"

visited_urls = set()

async def fetch_html(session, url, retries=3):
    for attempt in range(1, retries+1):
        try:
            # set a 30-second timeout and force aiohttp to close the connection
            timeout = aiohttp.ClientTimeout(total=30)
            async with session.get(url, timeout=timeout) as resp:
                resp.raise_for_status()
                return await resp.text()
        except (aiohttp.ServerDisconnectedError, aiohttp.ClientError) as e:
            print(f"[{attempt}/{retries}] error fetching {url}: {e}")
            if attempt == retries:
                return ""      # give up and return empty
            await asyncio.sleep(1)  # back off before retry

async def crawl(session, url, base_url, text_content, image_urls):
    if url in visited_urls:
        return
    visited_urls.add(url)

    html = await fetch_html(session, url)
    if not html:
        return

    soup = BeautifulSoup(html, "html.parser")
    for text in soup.stripped_strings:
        text_content.append(text)
    for img in soup.find_all("img", src=True):
        image_urls.append(urljoin(url, img['src']))

    for a in soup.find_all("a", href=True):
        link = urljoin(url, a['href'])
        if urlparse(link).netloc == urlparse(base_url).netloc:
            await crawl(session, link, base_url, text_content, image_urls)

async def main(start_url):
    text_content, image_urls = [], []
    # force_close=True makes each TCP connection be torn down promptly
    connector = aiohttp.TCPConnector(force_close=True, enable_cleanup_closed=True)
    headers = {"User-Agent": os.environ["USER_AGENT"]}

    async with aiohttp.ClientSession(headers=headers, connector=connector) as session:
        await crawl(session, start_url, start_url, text_content, image_urls)

    with open("output2.txt", "w", encoding="utf-8") as f:
        f.write("Text Content:\n" + "\n".join(text_content))
        f.write("\n\nImage URLs:\n" + "\n".join(image_urls))
    print("Done.")

if __name__ == "__main__":
    asyncio.run(main("https://www.occamsadvisory.com/"))


[1/3] error fetching https://www.occamsadvisory.com/tax-credit: 404, message='Not Found', url='https://www.occamsadvisory.com/tax-credit'
[2/3] error fetching https://www.occamsadvisory.com/tax-credit: 404, message='Not Found', url='https://www.occamsadvisory.com/tax-credit'
[3/3] error fetching https://www.occamsadvisory.com/tax-credit: 404, message='Not Found', url='https://www.occamsadvisory.com/tax-credit'
[1/3] error fetching https://www.occamsadvisory.com/terms-conditions/: 404, message='Not Found', url='https://www.occamsadvisory.com/terms-conditions'
[2/3] error fetching https://www.occamsadvisory.com/terms-conditions/: 404, message='Not Found', url='https://www.occamsadvisory.com/terms-conditions'
[3/3] error fetching https://www.occamsadvisory.com/terms-conditions/: 404, message='Not Found', url='https://www.occamsadvisory.com/terms-conditions'
Done.


In [2]:
!pip install langchain
!pip install openai
!pip install PyPDF2
!pip install faiss-cpu
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.7 MB/s eta 0:00:00


In [3]:
pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.8 MB/s eta 0:00:00


In [34]:
from PyPDF2 import PdfReader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your key"
os.environ["SERPAPI_API_KEY"] = "your key"

In [ ]:
# Open and read the content of a text file
file_path = '/content/output.txt'
with open(file_path, 'r') as file:
    raw_text = file.read()

# Output the raw text
raw_text


'Text Content:\nOccams Advisory | Global Financing Advisory & Professional Services\nAbout\nServices\nBSGI\nBusiness Services & Growth Incubation\nFTPS\nFinancial Technology & Payment Solutions\nCMIB\nCapital Markets & Investment Banking\nTC\nTax Credits\nStructure, Incorporation &\nAccounting\n                                                        Advisory\nBeneficial Ownership Information\n                                                        Report\nProcess Efficiency, Compliance,\nTax\n                                                        Planning & Filing\nBrand Building, Mobile Marketing &\nData\n                                                        Analytics\nDigital\n                                                        Presence & Social Media\nInformation\n                                                        Technology Services\nMerchant\n                                                        Accounts Across the Globe\nTailored Payment Solutions &\nVerification\n 

In [44]:
# We need to split the text using Character Text Split such that it sshould not increse token size
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 800,
    chunk_overlap  = 200,
    length_function = len,
)
texts = text_splitter.split_text(raw_text)

In [45]:
len(texts)

5363

In [46]:
# Download embeddings from OpenAI
embeddings = OpenAIEmbeddings()

In [47]:
document_search = FAISS.from_texts(texts, embeddings)

In [48]:
from langchain.chains.question_answering import load_qa_chain
from langchain.llms import OpenAI

In [49]:
chain = load_qa_chain(OpenAI(), chain_type="stuff")

<ipython-input-49-25e042d5cc81>:1: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  chain = load_qa_chain(OpenAI(), chain_type="stuff")
<ipython-input-49-25e042d5cc81>:1: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: 

In [50]:
# query = "	Tell me about Custom Software Development"
query = "	Tell me about occams advisory"
docs = document_search.similarity_search(query)
chain.run(input_documents=docs, question=query)

<ipython-input-50-990aa2fa83f0>:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chain.run(input_documents=docs, question=query)


" Occams Advisory is a company that was created to serve the small business market and address the most pressing needs of entrepreneurs. The company is named after the principle of Occams Razor, which states that the simplest answer is often correct. This philosophy is reflected in the company's approach to problem-solving, as they strive to provide simple solutions for their clients. Occams Advisory was founded on the idea of using this spirit of problem-solving to help small businesses thrive. "

In [61]:
# query = "	Tell me about Custom Software Development"
# query = "	who is the ceo of occams advisory"
query = "Where is Occams Advisory’s corporate head office located?"
docs = document_search.similarity_search(query)
chain.run(input_documents=docs, question=query)

' The corporate head office of Occams Advisory is located at 2170 Main St, Ste 203, Sarasota, FL 34237.'

In [ ]:
# from PyPDF2 import PdfReader
# from langchain.embeddings.openai import OpenAIEmbeddings
# from langchain.text_splitter import CharacterTextSplitter
# from langchain.vectorstores import FAISS
# import os

# #set api keys
# os.environ["OPENAI_API_KEY"] = "api key"
# os.environ["SERPAPI_API_KEY"] = "api key"

# # Open and read the content of a text file
# file_path = '/content/output1.txt'
# with open(file_path, 'r') as file:
#     raw_text = file.read()

# # Output the raw text (optional for debugging)
# print(raw_text)

# # Split the text using CharacterTextSplitter to avoid large token sizes
# text_splitter = CharacterTextSplitter(
#     separator="\n",
#     chunk_size=800,
#     chunk_overlap=200,
#     length_function=len,
# )
# texts = text_splitter.split_text(raw_text)

# # Download embeddings from OpenAI
# embeddings = OpenAIEmbeddings()

# # Create FAISS vector store from the texts
# document_search = FAISS.from_texts(texts, embeddings)

# # Load the question-answering chain with a stricter answer generation process
# from langchain.chains.question_answering import load_qa_chain
# from langchain.llms import OpenAI

# chain = load_qa_chain(OpenAI(), chain_type="map_reduce")  # 'map_reduce' to ensure focused responses

# # Function to answer a query strictly based on the document content
# def get_answer_from_document(query):
#     # Perform a similarity search for the most relevant documents
#     docs = document_search.similarity_search(query, k=3)  # Retrieve top 3 documents for better relevance
#     answer = chain.run(input_documents=docs, question=query)
#     return answer

# # Example query
# query = "Tell me about Occams Advisory"
# answer = get_answer_from_document(query)

# # Output the answer based only on document content
# print(answer)


Text Content:
Occams Advisory | Global Financing Advisory & Professional Services
About
Services
BSGI
Business Services & Growth Incubation
FTPS
Financial Technology & Payment Solutions
CMIB
Capital Markets & Investment Banking
TC
Tax Credits
Structure, Incorporation &
Accounting
                                                        Advisory
Beneficial Ownership Information
                                                        Report
Process Efficiency, Compliance,
Tax
                                                        Planning & Filing
Brand Building, Mobile Marketing &
Data
                                                        Analytics
Digital
                                                        Presence & Social Media
Information
                                                        Technology Services
Merchant
                                                        Accounts Across the Globe
Tailored Payment Solutions &
Verification
                                 

<ipython-input-4-a798ff8e4d27>:29: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()
<ipython-input-4-a798ff8e4d27>:38: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  chain = load_qa_chain(OpenAI(), chain_type="map_reduce")  # 'map_reduce' to ensure focused responses
<ipython-input-4-a798ff8e4d27>:38: LangChainDeprecationWarning: This class is deprecated. See th

 Occams Advisory is a consulting firm that provides simple solutions to the most pressing needs of entrepreneurs in the small business market.


In [7]:
# Example query
# query = "What is Occams Advisory’s stated mission and values?"
query = "Where is Occams Advisory’s corporate head office located?"
answer = get_answer_from_document(query)

# Output the answer based only on document content
print(answer)

 Occams Advisory's corporate head office is located at 2170 Main St, Ste 203, Sarasota, FL 34237.


In [ ]:
from PyPDF2 import PdfReader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
import os

#set api keys
os.environ["OPENAI_API_KEY"] = "your key"
os.environ["SERPAPI_API_KEY"] = "your key"

# Open and read the content of a text file
file_path = '/content/output.txt'
with open(file_path, 'r') as file:
    raw_text = file.read()

# Output the raw text (optional for debugging)
print(raw_text)

# Split the text using CharacterTextSplitter to avoid large token sizes
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,  # Increased chunk size to allow larger portions of text
    chunk_overlap=200,
    length_function=len,
)
texts = text_splitter.split_text(raw_text)

# Download embeddings from OpenAI
embeddings = OpenAIEmbeddings()

# Create FAISS vector store from the texts
document_search = FAISS.from_texts(texts, embeddings)

# Load the question-answering chain with a stricter answer generation process
from langchain.chains.question_answering import load_qa_chain
from langchain.llms import OpenAI

# Increase max tokens to allow larger outputs
chain = load_qa_chain(OpenAI(max_tokens=2000), chain_type="map_reduce")  # Increased max_tokens to allow longer answers

# Function to answer a query strictly based on the document content
def get_answer_from_document(query):
    # Perform a similarity search for the most relevant documents
    docs = document_search.similarity_search(query, k=5)  # Increased k to get more relevant documents
    answer = chain.run(input_documents=docs, question=query)
    return answer

# Example query
query = "Which awards did Occams Advisory receive in 2024?"
answer = get_answer_from_document(query)

# Output the answer based only on document content
print(answer)


Streaming output truncated to the last 5000 lines.
Priyanka Bathwal
AVP - Finance & Accounting
Read More
Sairaj Adate
Software Engineer - Web Development
Read More
Md Ashfaq Akhtar
Frontend Developer - Web Development
Read More
| Follow on Linked In
Sairaj Adate
Software Engineer - Web Development
Sairaj is a passionate Frontend and Web Developer with over two years of experience at Occams Advisory. He leverages his expertise in PHP, JavaScript, and WordPress to design and build engaging, user-friendly websites.
Skilled in crafting intuitive user interfaces and optimizing website performance, Sairaj thrives on challenges, tackling everything from custom WordPress themes to complex JavaScript features. His dedication lies in creating impactful digital experiences.
| Follow on Linked In
Md Ashfaq Akhtar
Frontend Developer - Web Development
Ashfaq Akhtar is our Frontend Web Developer with a passion for building user-friendly and visually appealing web experiences. Over a year of experienc

In [17]:
# Example query
# query = "Which awards did Occams Advisory receive in 2024?"
# query = "Can you name three specific offerings under the Tax Credits (TC) vertical?"
query = "What Sell-Side M&A services are offered by Occams Advisory?"
answer = get_answer_from_document(query)

# Output the answer based only on document content
print(answer)

 Occams Advisory offers Sell-Side M&A services, including due diligence, deal structuring, negotiations, and post-merger integration.
